
#การแก้ไขชั่วคราว

การแก้ไขชั่วคราว inference โดยใช้โมเดล InterpModAFNO

ตัวอย่างนี้สาธิตวิธีการใช้โมเดล InterpModAFNO เพื่อประมาณค่า
forecast จากโมเดลพื้นฐานไปจนถึงความละเอียดเวลาที่ละเอียดยิ่งขึ้น
โมเดลพยากรณ์โรคที่มีอยู่หลายตัวมีขนาดขั้นเป็น 6 ชั่วโมง ซึ่งอาจพิสูจน์ได้
ไม่เพียงพอสำหรับบางแอปพลิเคชัน
InterpModAFNO มอบวิธีการขับเคลื่อนด้วย AI สำหรับการรับความละเอียดรายชั่วโมงโดยให้เวลา 6 ชั่วโมง
การคาดการณ์

ในตัวอย่างนี้คุณจะได้เรียนรู้:

- วิธีโหลดฐาน Prognostic Model
- วิธีโหลดโมเดล InterpModAFNO
- วิธีการรันโมเดลการแก้ไข
- วิธีพล็อตผลลัพธ์


In [ ]:
# /// script
# dependencies = [
#   "torch==2.9.1", # Match lock file to avoid torch-harmonics issue
#   "earth2studio[sfno,interp-modafno] @ git+https://github.com/NVIDIA/earth2studio.git",
#   "matplotlib",
# ]
# ///

## การเตรียมองค์ประกอบ
ขั้นแรก นำเข้าโมดูลที่จำเป็นและตั้งค่าสภาพแวดล้อมของเราและโหลดโมเดล
เราจะใช้ SFNO เป็นฐาน Prognostic Model และโมเดล InterpModAFNO
สอดแทรกเอาต์พุตเพื่อให้ได้ความละเอียดเวลาที่ละเอียดยิ่งขึ้น
Prognostic Model ต้องทำนายตัวแปรที่จำเป็นในโมเดลการแก้ไข

ตัวอย่างนี้ต้องใช้องค์ประกอบต่อไปนี้:

- รุ่นการแก้ไข: :py:class:`earth2studio.models.px.InterpModAFNO`.
- โมเดลพยากรณ์พื้นฐาน: ใช้รุ่น SFNO :py:class:`earth2studio.models.px.SFNO`
- Datasource: ดึงข้อมูลจาก GFS data API ผ่าน :py:class:`earth2studio.data.GFS`.



In [ ]:
import os

import matplotlib.pyplot as plt

from earth2studio.data import GFS
from earth2studio.io import ZarrBackend
from earth2studio.models.px import SFNO, InterpModAFNO

# สร้างไดเรกทอรีผลลัพธ์
os.makedirs("outputs", exist_ok=True)

sfno_package = SFNO.load_default_package()
px_model = SFNO.load_model(sfno_package)

# โหลดโมเดลการแก้ไข
interp_package = InterpModAFNO.load_default_package()
interp_model = InterpModAFNO.load_model(interp_package, px_model=px_model)

# สร้างแหล่งข้อมูล
data = GFS()

# สร้างตัวจัดการ IO
io = ZarrBackend()

## เรียกใช้โมเดลการแก้ไข
ตอนนี้รันโมเดลการแก้ไขเพื่อรับ forecasts ด้วยความละเอียดเวลาที่ละเอียดยิ่งขึ้น
โมเดลพื้นฐาน (SFNO) สร้าง forecast ในช่วงเวลา 6 ชั่วโมง และ
โมเดลการประมาณค่าจะประมาณช่วง 1 ชั่วโมง



In [ ]:
# กำหนดพารามิเตอร์ forecast
forecast_date = "2024-01-01"
nsteps = 5  # จำนวนของขั้นตอน forecast ที่ประมาณค่าไว้

# เรียกใช้โมเดล
from earth2studio.run import deterministic

io = deterministic([forecast_date], nsteps, interp_model, data, io)

print(io.root.tree())

## เห็นภาพผลลัพธ์
ลองนึกภาพไอน้ำรวมคอลัมน์ (tcwv) ในแต่ละขั้นตอน
และบันทึกเป็นไฟล์แยกกัน



In [ ]:
# รับจำนวนก้าวของเวลา
n_steps = io["tcwv"].shape[1]

# สร้างร่างเดียวพร้อมโครงเรื่องย่อย
fig, axs = plt.subplots(2, 3, figsize=(15, 6))
axs = axs.ravel()

# สร้างแปลงสำหรับแต่ละขั้นตอนเวลา
for step in range(min([n_steps, 6])):
    im = axs[step].imshow(
        io["tcwv"][0, step], cmap="twilight_shifted", aspect="auto", vmin=0, vmax=85
    )
    axs[step].set_title(f"Water Vapour - Step: {step} hrs")
    fig.colorbar(im, ax=axs[step], label="kg/m^2")

plt.tight_layout()
# บันทึกรูป
plt.savefig("outputs/12_tcwv_steps.jpg")